In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
spark=SparkSession.builder.appName("Upi_trans_silver").getOrCreate()

In [0]:
schema = StructType([
    StructField("txn_id", StringType(), True),
    StructField("rrn", StringType(), True),
    StructField("payer_vpa", StringType(), True),
    StructField("payee_vpa", StringType(), True),
    StructField("payer_bank", StringType(), True),
    StructField("payee_bank", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("txn_type", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("status", StringType(), True),
    StructField("npci_response_code", StringType(), True),
    StructField("initiated_timestamp", StringType(), True)
    ])

In [0]:
## INflating the json, casting and aliasing

bronze_df =(spark.readStream.table("workspace.default.upi_transactions_bronze")
    .select(
    col("key"),from_json(col("value"),schema).alias("fields"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")
    )
    .select(
    col("key").cast("string").alias("key"),
    col("fields.txn_id").cast("string").alias("txn_id"),
    col("fields.rrn").cast("string").alias("rrn"),
    col("fields.payer_vpa").cast("string").alias("payer_vpa"),
    col("fields.payee_vpa").cast("string").alias("payee_vpa"),
    col("fields.payer_bank").cast("string").alias("payer_bank"),
    col("fields.payee_bank").cast("string").alias("payee_bank"),
    col("fields.amount").cast("decimal(10,2)").alias("amount"),
    col("fields.txn_type").cast("string").alias("txn_type"),
    col("fields.channel").cast("string").alias("channel"),
    col("fields.status").cast("string").alias("status"),
    col("fields.npci_response_code").cast("string").alias("npci_response_code"),
    col("fields.initiated_timestamp").cast("timestamp").alias("initiated_timestamp"),
    col("topic").cast("string").alias("topic"),
    col("partition").cast("int").alias("partition"),
    col("offset").cast("long").alias("offset"),
    col("timestamp").cast("timestamp").alias("timestamp"),
    col("timestampType").cast("int").alias("timestampType")
    )
)

bronze_query = (
                bronze_df.writeStream
                .format("delta")
                .outputMode("append")
                .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/bronze_inflated/")
                .trigger(availableNow = True)
                .queryName("bronze_query")
                .toTable("workspace.default.upi_transactions_bronze_inflated")
                
                )


In [0]:

bronze_df = (bronze_df.withColumn("late_flag",when(col("initiated_timestamp") < current_timestamp() - expr("interval 2 minute"),lit(1)).otherwise(lit(0)))
        .withColumn("txn_date",date_format(col("initiated_timestamp"),"yyyy-MM-dd"))
       .withColumn("txn_hour",date_format(col("initiated_timestamp"),"HH")))

column_validations = (
                        col("txn_id").isNotNull() 
                        & col("rrn").isNotNull() 
                        & col("amount").isNotNull() 
                        & col("initiated_timestamp").isNotNull()
                        & ((col("status") == "PENDING")
                            | col("npci_response_code").isNotNull()
                          )
                        & ((col("amount") > 0) & (col("amount")<100000))
                        & col("status").isin(["SUCCESS", "FAILED", "PENDING"])
                        & col("channel").isin(["APP", "QR", "COLLECT"])
                        & col("txn_type").isin(["P2P", "P2M"])
                    )
                    
quarantine_df = bronze_df.filter(~column_validations)

quarantine_query = (quarantine_df.writeStream.format("delta")
                        .outputMode("append")
                        .option("checkpointLocation","/Volumes/workspace/default/checkpoints/Quarantine/")
                        .trigger(availableNow = True)
                        .queryName("quarantine_query")
                        .toTable("workspace.default.upi_transactions_quarantine")
)


## Validating wheteher important columns are there or not
column_validated_df = bronze_df.filter(column_validations)


column_standardizations_df = (column_validated_df
                              .withColumn("status",upper(col("status")))
                              .withColumn("txn_type",upper(col("txn_type")))
                              .withColumn("channel",upper(col("channel")))
                              .withColumn("payer_vpa",lower(col("payer_vpa")))
                              .withColumn("payee_vpa",lower(col("payee_vpa")))
                              )

query_silver = (column_standardizations_df.writeStream
               .format("delta")
               .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/Silver")
               .outputMode("append")
               .trigger(availableNow = True)
               .queryName("inflated_query")
               .toTable("workspace.default.upi_transactions_standardized")
               )


In [0]:
silver_dedup = (spark.readStream.table("workspace.default.upi_transactions_standardized")
                          .withWatermark("initiated_timestamp", "20 minutes")
                          .dropDuplicatesWithinWatermark(["txn_id"])
                          .withColumn("silver_processed_timestamp",current_timestamp())
)
query_dedup = (silver_dedup.writeStream
               .format("delta")
               .option("checkpointLocation","/Volumes/workspace/default/checkpoints/silver_dedup/")
               .outputMode("append")
               .trigger(availableNow = True)
               .queryName("dedup_query")
               .toTable("workspace.default.upi_transactions_silver")
)

In [0]:
%sql
create or replace view silver_view as
select * from upi_transactions_silver where status ='PENDING' or status ='SUCCESS'

> # **Settlement_data enrichments**

In [0]:
%sql
select * from settlement_data

In [0]:
settlement_bronze_df = spark.readStream.table("workspace.default.settlement_data")

In [0]:
column_validations = (
                        col("txn_id").isNotNull() 
                        & col("rrn").isNotNull() 
                        & col("settlement_amount").isNotNull() 
                        & col("settled_timestamp").isNotNull()
                        & ((col("settlement_amount") > 0) & (col("settlement_amount")<100000))
                        & col("settlement_status").isin(["SUCCESS", "PENDING"])
                    )

In [0]:
settlement_bronze_df = (settlement_bronze_df.filter(column_validations)
                        .withColumn("settlement_status",upper(col("settlement_status")))
                        .withWatermark("settled_timestamp", "1 day")
                        .dropDuplicatesWithinWatermark(["txn_id"])
                        )
        
settlement_silver_query = (settlement_bronze_df.writeStream
                           .format("delta")
                           .outputMode("append")
                           .option("checkpointLocation","/Volumes/workspace/default/checkpoints/settlement_silver/")
                           .trigger(availableNow = True)
                           .queryName("settlement_silver_query")
                           .toTable("workspace.default.settlement_silver")
                            )


In [0]:
settlement_quarantine = settlement_bronze_df.filter(~column_validations)

query_quarantine = (settlement_quarantine.writeStream
                     .format("delta")
                     .outputMode("append")
                     .option("checkpointLocation","/Volumes/workspace/default/checkpoints/settlement_quarantine/")
                     .trigger(availableNow = True)
                     .queryName("settlement_quarantine_query")
                     .toTable("workspace.default.settlement_quarantine")
)